# License Plate Recognition - Training

In [1]:
# Cross-Platform Environment Setup
import os
import sys

def setup_environment():
    """Detects platform and sets up the environment."""
    in_colab = 'google.colab' in sys.modules
    in_kaggle = os.environ.get('KAGGLE_URL_BASE') is not None
    
    if in_colab or in_kaggle:
        print(f"Running on {'Google Colab' if in_colab else 'Kaggle'}. Installing dependencies...")
        !pip install -qU ultralytics wandb roboflow python-dotenv supervision easyocr cvzone
        
        if in_colab:
            from google.colab import userdata
            os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY') or ''
            os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY') or ''
        elif in_kaggle:
            try:
                from kaggle_secrets import UserSecretsClient
                user_secrets = UserSecretsClient()
                os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY')
                os.environ['ROBOFLOW_API_KEY'] = user_secrets.get_secret('ROBOFLOW_API_KEY')
            except Exception:
                print("Kaggle secrets not found. Please set them in the Add-ons menu.")
    else:
        print("Running locally. Loading environment...")
        try:
            from dotenv import load_dotenv
            load_dotenv('../.env')
        except ImportError:
            print("python-dotenv not found. Install it with: pip install python-dotenv")

    # Verify keys
    if not os.environ.get('WANDB_API_KEY'):
        print("Warning: WANDB_API_KEY not set.")
    if not os.environ.get('ROBOFLOW_API_KEY'):
        print("Warning: ROBOFLOW_API_KEY not set.")

setup_environment()

Running locally. Loading environment...


## 1. W&B & Dataset Setup

In [2]:
import wandb
from roboflow import Roboflow
from ultralytics import YOLO

# Login to W&B
if os.environ.get('WANDB_API_KEY'):
    wandb.login(key=os.environ.get('WANDB_API_KEY'))

# # Download Dataset
# rf = Roboflow(api_key=os.environ.get("ROBOFLOW_API_KEY"))
# project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
# version = project.version(11)
# dataset = version.download("yolov8", location='../datasets')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/ziadmoh/.netrc
wandb: Currently logged in as: ziadmohamedgamal25 (ziadmohamedgamal25-suez-canal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
# ── Training Config ──────────────
EPOCHS = 5
IMGSZ  = 640
BATCH  = 16
MODEL_TYPE = "yolov8n.pt"
PROJECT_NAME = "License_Plate_Detection_V1"
RUN_NAME = "v2_Plate_detection"

run = wandb.init(
    entity="ahmed-hossam-suez-canal-university",
    project=PROJECT_NAME,
    name=RUN_NAME,
    job_type="training",
    config = {
        "model":        MODEL_TYPE,
        "pretrained":   True,
        "epochs":       EPOCHS,
        "imgsz":        IMGSZ,
        "batch":        BATCH,
        "fraction":     1.0,
        "dataset":      "license-plate-recognition-rxg4e",
    }
)
print(f"W&B run started: {run.url}")

W&B run started: https://wandb.ai/ahmed-hossam-suez-canal-university/License_Plate_Detection_V1/runs/7vhhbro9


## 2. Modeling

In [4]:
import glob
cfg = wandb.config
model = YOLO(cfg.model)

results = model.train(
    project='../experiments/runs/detect',
    data=glob.glob('../datasets/License-Plate-Recognition-*/data.yaml')[0],
    epochs=cfg.epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,
    fraction=cfg.fraction,
    name=RUN_NAME,
    plots=True,
)

New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.8 🚀 Python-3.11.13 torch-2.9.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 5806MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/License-Plate-Recognition-11/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic

In [5]:
# ── W&B: Log final validation metrics ─────────────────────────────────
metrics_dict = results.results_dict

final_metrics = {
    "final/precision":    metrics_dict.get("metrics/precision(B)", 0),
    "final/recall":       metrics_dict.get("metrics/recall(B)",    0),
    "final/mAP50":        metrics_dict.get("metrics/mAP50(B)",     0),
    "final/mAP50-95":     metrics_dict.get("metrics/mAP50-95(B)",  0),
    "final/box_loss":     metrics_dict.get("val/box_loss",         0),
    "final/cls_loss":     metrics_dict.get("val/cls_loss",         0),
    "final/dfl_loss":     metrics_dict.get("val/dfl_loss",         0),
    "final/fitness":      results.fitness,
}
wandb.log(final_metrics)

# Also write them as W&B summary so they show in the runs table
for k, v in final_metrics.items():
    wandb.run.summary[k] = v

print("Logged metrics:")
for k, v in final_metrics.items():
    print(f"  {k}: {v:.4f}")


Logged metrics:
  final/precision: 0.9696
  final/recall: 0.9308
  final/mAP50: 0.9635
  final/mAP50-95: 0.6648
  final/box_loss: 0.0000
  final/cls_loss: 0.0000
  final/dfl_loss: 0.0000
  final/fitness: 0.6648


In [6]:
# ── W&B: Log training plots & validation images ───────────────────────
from pathlib import Path

save_dir = Path(results.save_dir)

plot_files = {
    "confusion_matrix":           save_dir / "confusion_matrix.png",
    "confusion_matrix_normalized": save_dir / "confusion_matrix_normalized.png",
    "BoxPR_curve":                   save_dir / "BoxPR_curve.png",
    "BoxF1_curve":                   save_dir / "BoxF1_curve.png",
    "BoxP_curve":                    save_dir / "BoxP_curve.png",
    "BoxR_curve":                    save_dir / "BoxR_curve.png",
    "results":                    save_dir / "results.png",
    "labels":                     save_dir / "labels.jpg"
}

wandb_images = {}
for name, path in plot_files.items():
    if path.exists():
        wandb_images[f"plots/{name}"] = wandb.Image(str(path), caption=name)
        print(f"  ✓ {name}")
    else:
        print(f"  ✗ {name} not found")

# Validation batch predictions (ground-truth vs predictions)
for img_path in sorted(save_dir.glob("val_batch*.jpg")):
    wandb_images[f"val_batches/{img_path.stem}"] = wandb.Image(
        str(img_path), caption=img_path.stem
    )
    print(f"  ✓ {img_path.stem}")

wandb.log(wandb_images)
print(f"\nLogged {len(wandb_images)} images/plots to W&B.")


  ✓ confusion_matrix
  ✓ confusion_matrix_normalized
  ✓ BoxPR_curve
  ✓ BoxF1_curve
  ✓ BoxP_curve
  ✓ BoxR_curve
  ✓ results
  ✓ labels
  ✓ val_batch0_labels
  ✓ val_batch0_pred
  ✓ val_batch1_labels
  ✓ val_batch1_pred
  ✓ val_batch2_labels
  ✓ val_batch2_pred

Logged 14 images/plots to W&B.


In [7]:
# ── W&B: Save best model as a versioned artifact ──────────────────────
best_pt = save_dir / "weights" / "best.pt"
last_pt = save_dir / "weights" / "last.pt"

artifact = wandb.Artifact(
    name=f"{RUN_NAME}_model",
    type="model",
    description="YOLOv8n fine-tuned",
    metadata={
        "mAP50":     wandb.run.summary.get("final/mAP50"),
        "mAP50-95":  wandb.run.summary.get("final/mAP50-95"),
        "precision": wandb.run.summary.get("final/precision"),
        "recall":    wandb.run.summary.get("final/recall"),
        "epochs":    cfg.epochs,
        "imgsz":     cfg.imgsz,
    }
)

if best_pt.exists():
    artifact.add_file(str(best_pt), name="best.pt")
if last_pt.exists():
    artifact.add_file(str(last_pt), name="last.pt")

wandb.log_artifact(artifact)
artifact.wait()  # ← block until artifact is fully logged on the W&B server
print(f"Model artifact logged: {artifact.name}:{artifact.version}")


Model artifact logged: v2_Plate_detection_model:v0:v0


In [8]:
wandb.finish()

final/box_loss,▁
final/cls_loss,▁
final/dfl_loss,▁
final/fitness,▁
final/mAP50,▁
final/mAP50-95,▁
final/precision,▁
final/recall,▁
final/box_loss,0
final/cls_loss,0
final/dfl_loss,0
